---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-09: Developing LLM Apps using Gradio</h1>

# Learning agenda of this notebook
<h3 align="left" style="color: purple;">1. What is Gradio?</h3>

- Gradio Example 1 (Hello Gradio)
- Gradio Example 2 (Using Slider)
- Gradio Example 3 (Using Dropdown)
- Gradio Example 4 (Using Multi-Line Input and Output)
- Gradio Example 5 (Using File Upload Component)
- Gradio Example 6: Q/A Bot using OpenAI's Responses API (Local/Cloud Models via Ollama)
- Gradio Example 7: Q/A Bot using OpenAI's Responses API (Groq Hosted Models)
- Gradio Example 8: A Q/A Bot using Llama3.2 running on Local Box and Claude Haiku 4.5
- Gradio Example 9: A Q/A Bot that Answers from a File Contents (Local/Cloud Models via Ollama)
- Gradio Example 10: A Research Paper Summarizer (Local/Cloud Models via Ollama)
- Gradio Example 11: A Q/A Bot that Extracts a youtube video transcript and Answers from the Contents (Local/Cloud Models via Ollama)
- Gradio Example 12: A Q/A Bot that Answers from a Webpage (Local/Cloud Models via Ollama)

<h3 align="left" style="color: purple;">2. Programming Assignment 01</h3>

<h1 align="center" style="color: red;">
Students must practice running code examples in this Notebook with models hosted on 
<a href="https://platform.openai.com/docs/models" target="_blank" style="color: green; text-decoration: none;">OpenAI, </a> 
<a href="https://www.anthropic.com/" target="_blank" style="color: green; text-decoration: none;">Anthropic, </a> 
<a href="https://huggingface.co/models" target="_blank" style="color: green; text-decoration: none;">Hugging Face, </a> 
<a href="https://console.groq.com/docs/models" target="_blank" style="color: green; text-decoration: none;">Groq, </a> and 
<a href="https://ollama.com/" target="_blank" style="color: green; text-decoration: none;">Ollama</a> 
using OpenAI-compatible APIs such as 
<span style="color: purple; font-family: 'Courier New', monospace; font-weight: bold;">
<a href="https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create" target="_blank" style="color: inherit; text-decoration: none;">chat.completions.create()</a>
</span> 
and 
<span style="color: purple; font-family: 'Courier New', monospace; font-weight: bold;">
<a href="https://platform.openai.com/docs/api-reference/responses" target="_blank" style="color: inherit; text-decoration: none;">responses.create()</a>
</span> 
by configuring the correct <span style="color: darkred;">base_url or API endpoints</span>.
</h1>

- **`base_url` Parameter:** Every Provider has its own base URL, the following base urls are compatible with OpenAI client less Anthropic:


| Provider              | Base URL                                                  | Will it work with OpenAI client `chat.completions.create()`? | Will it work with OpenAI client `responses.create()`? | Notes                                                                                                                                                                                 |
| --------------------- | --------------------------------------------------------- | ------------------------------------------------------------ | ----------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **OpenAI**            | `https://api.openai.com/v1`                               | ✅ Yes                                                        | ✅ Yes                                                 | Native OpenAI API. Fully compatible.                                                                                                                                                  |
| **Anthropic**         | `https://api.anthropic.com/v1`                            | ❌ No                                                         | ❌ No                                                  | Anthropic's API is **not OpenAI-compatible** — neither `chat.completions.create()` nor `responses.create()` works against this base URL. Use the native `anthropic` SDK (`client.messages.create()`), as done in Gradio Example 8. |
| **Google (Gemini API)** | `https://generativelanguage.googleapis.com/v1beta/openai` | ✅ Yes                                                        | ✅ Yes                                                 | Google exposes OpenAI-compatible endpoints, so `chat.completions.create()` and `responses.create()` work.                                                                             |
| **Groq**              | `https://api.groq.com/openai/v1`                          | ✅ Yes                                                        | ✅ Yes                                                 | Groq exposes OpenAI-compatible endpoints for both chat and responses.                                                                                                                 |
| **Ollama**            | `http://localhost:11434/v1`                               | ✅ Yes                                                        | ✅ Yes                                                 | Local Ollama server supports OpenAI-compatible endpoints.                                                                                                                             |
| **Deepseek**          | `https://api.deepseek.com/v1`                             | ✅ Yes                                                        | ✅ Yes                                                 | Deepseek exposes OpenAI-compatible endpoints.                                                                                                                                         |


# <span style='background :lightgreen' >1. What is Gradio? (https://www.gradio.app/)</span>
- **Gradio** is an open-source Python library that lets you quickly create interactive web interfaces for machine learning models.
- It provides simple UI components (textboxes, dropdown, checkbox, slider, file uploads, images, etc.) that can wrap any Python function with just a few lines of code.
- No frontend knowledge is required—Gradio automatically builds a responsive web interface.
- You can test models locally or share them publicly via temporary shareable links.
- Supports real-time streaming, async functions, and integration with popular frameworks (Hugging Face, PyTorch, TensorFlow, OpenAI/Ollama APIs).
- Ideal for demos, prototyping, teaching, research, and showcasing ML models.
- When you run Gradio, it runs a server in the background at your local machine on port **7860** by default (if that port is busy it tries 7861, 7862, ... until it finds a free one)

### Ways to Build Gradio Interfaces
#### a. **`gradio.Interface`:**
- Constructor used to build an *basic interactive UI* by binding your Python function to customizable input/output components
- Stateless by default (no built-in memory)
- Requires explicit input/output components
- Automatically creates layout for you
- Good for:
    - Single Q&A
    - Predictions
    - Text → Text tasks
    - Simple demos
#### b. **`gradio.Blocks`:**
- Most flexible way to build custom UIs
- Allows combining multiple inputs, outputs & logic
- Supports event-driven apps (buttons, uploads, etc.)
- Use `gr.Row()` to arrange components horizontally, and `gr.Column()` to arrange components vertically
- You can nest rows and columns, e.g., keeping inputs on the left and outputs on the right
- Unlike `Interface`, `Blocks` supports:
    - Button clicks → `.click()`
    - Text change → `.change()`
    - File upload → `.upload()`
    - Page load → `.load()`
- Needed when:
    - You want multi-step workflows
    - You want multiple outputs
    - You want full control over layout
- Good for:
    - AI tools
    - Dashboards
    - Agent apps
    - Research assistants

#### c. **`gradio.ChatInterface`:**
- Constructor used to build a *conversational chatbot UI* with pre-configured chat components and automatic message history management
- Specialized for conversational AI apps and any multi-turn dialogue system where context and conversation flow are essential
- Stateful (automatically manages chat history)
- Function MUST accept `(message, history)`
- Built-in chat UI (message bubbles, auto-scroll, etc.)
- No manual layout needed
- Good for:
    - Chatbots
    - Assistants
    - Multi-turn conversations

<h3 align="center"><div class="alert alert-success" style="margin: 20px">The `.launch()` method starts a local web server to make your Gradio app accessible in a browser,  with optional public sharing capabilities.</div></h3>

# <span style='background :lightgreen' >Gradio Example 1: Hello Gradio</span>
- When a user types a name into the input textbox and submits it (either by pressing Enter or clicking a submit button that Gradio automatically creates), the interface sends that text as an argument to the greet function, which concatenates "Hello " with the provided name and an exclamation mark, then returns this greeting string back to the interface, where it's immediately displayed in the output textbox below the input field. The flagging_mode="never" parameter disables the flagging feature, so users won't see any buttons to flag or save outputs for review.

In [3]:
##########################     USING PYTHON INPUT FUNCTION ##########################
def greet(name: str, age: int) -> str:
    return f"Hello Mr. {name}. You are {age} years old!"

name = input("Enter your name: ")
age = int(input("Enter your age: "))
print(greet(name, age))

Enter your name:  Arif Butt
Enter your age:  58


Hello Mr. Arif Butt. You are 58 years old!


In [4]:
##########################     USING `gradio.Interface() Method ##########################
import gradio as gr

def greet(name: str, age: int) -> str:
    return f"Hello Mr. {name}. You are {age} years old!"
    #return "Hello Mr. " + name + ". You are " + str(age) + " years old!"
    
demo = gr.Interface(
                    fn=greet,                                       # REQUIRED — function that runs when user submits input
                    inputs=[                                        # REQUIRED — input components (textbox, html, number, image, audio, dropdown, checkbox, slider, file, etc.)
                            gr.Textbox(label="Enter your name", placeholder="Your name here...", autofocus=True),
                            gr.Number(label="Enter your age", value=0)
                            ],
                    outputs = [gr.Textbox(label="The message is:")], # REQUIRED — output components (textbox, image, audio, plot, html, etc.)
                    title="Hello Gradio",                            # OPTIONAL — title displayed at top of interface (default: None)
                    description="## Enter your name and age",        # OPTIONAL — interface description text under the title (default: None)
                    flagging_mode="manual",                          # OPTIONAL — "manual" (default): Shows a "Flag" button users can click. "never": No flagging button appears. "auto": Every interaction is automatically flagged
                    flagging_dir="my_flagged_data"                   # OPTIONAL - By default, Gradio saves flagged samples in dataset1.csv file in pwd, or inside the directory mentioned in this argument.
                    )
demo.launch(inbrowser=True, share=True) # Launches the Gradio app without opening a browser (inbrowser) and without generating a public shareable link (share)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://423e1e39197a7ab939.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
##########################     USING `gradio.Blocks() Method ##########################
import gradio as gr

def greet(name: str, age: int) -> str:
    return f"Hello Mr. {name}. You are {age} years old!"
    #return "Hello Mr. " + name + ". You are " + str(age) + " years old!"
# gr.Blocks is Gradio’s layout + app container system 
with gr.Blocks() as demo:    # Everything written inside the with block automatically becomes part of the same Gradio app
    gr.Markdown("<h1 style='text-align: center;'>Hello Gradio</h1>")
    gr.Markdown("## Enter your Name and Age")
    with gr.Row():
        with gr.Column():
            name_input = gr.Textbox(label="Enter your name",  placeholder="Your name here...", autofocus=True)
            age_input = gr.Textbox(label="Enter your age")
            send_button = gr.Button("Send", variant="primary")
        with gr.Column():
            output = gr.Textbox(label="The message is:")

    # It tells Gradio: “When this button is clicked → run a function → take values from these inputs → send result to these outputs” (Other events can be double click, hover, mouse down, mouse up, leave etc)
    send_button.click(
                    fn=greet,
                    inputs=[name_input, age_input],
                    outputs=[output]
                    )    
demo.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


In [23]:
##########################     USING `gradio.ChatInterface() Method ##########################
import gradio as gr

def chat_fn(message: str, history: list) -> str:
    return f"The message to be sent to AI is: {message}"
    #return "The message to be sent to AI is: " + message
    
view = gr.ChatInterface(
                    fn=chat_fn,                                                    # REQUIRED — function that runs when user submits input
                    title="Hello Gradio",                                          # OPTIONAL — title displayed at top of interface (default: None)
                    description="Enter your name and age and see what I can do?",  # OPTIONAL — interface description text under the title (default: None)
                    flagging_mode="manual",                                        # OPTIONAL — "manual" (default): Shows a "Flag" button users can click. "never": No flagging button appears. "auto": Every interaction is automatically flagged
                    flagging_dir="my_flagged_data"        # By default, Gradio saves flagged samples in ./flagged directory, which you can override by specifying the path of a directory
             )
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 2: Using Slider</span>

In [5]:
import gradio as gr
def myfunc(name:str, intensity:int=0) -> str:
    if not name.strip():
        return "❗ Paste enter your name..."
    return "Hello, " + name + "!" * int(intensity)

view = gr.Interface(
                    fn=myfunc, 
                    inputs=[                                                   
                            gr.Textbox(label="Enter your name", placeholder="Your name here..."),
                            gr.Slider(label="Enter the count", value=0)
                            ],
                    outputs=["text"],
                    title="Use of Slider in Gradio",   
                    flagging_mode="never"
                    )
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 3 Using Dropdown</span>

In [25]:
import gradio as gr

def myfunc(name:str, mood:str):
    return f"Hello, {name}! You are feeling {mood.lower()} today."

view = gr.Interface(
                fn=myfunc,
                inputs=[
                        gr.Textbox(label="Enter your name"),
                        gr.Dropdown(
                                    choices=["Happy", "Excited", "Relaxed", "Motivated"],
                                    label="Select your mood",
                                    value="Happy"   # default value
                                    )
                        ],
                outputs="text",
                title="Use of Dropdown in Gradio",  
                flagging_mode="never"
)

view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 4 Using Multi-Line Input and Output</span>

In [26]:
import gradio as gr

def myfunc(message):
    return "This is your message:\n" + message

view = gr.Interface(
                    fn=myfunc,
                    inputs=[gr.Textbox(label="Your message:", lines=6)],
                    outputs=[gr.Textbox(label="Response:", lines=8)],
                    description="Enter your multi-line message:",
                    title="Use of Multi-line I/O  in Gradio",                                          
                    flagging_mode="never"
)
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 5 Using File Upload Component</span>
- You may have to install the python-docx package using `uv add python-docx`

In [27]:
import gradio as gr
import docx  # for reading .docx files

def read_file(file_obj):
    if file_obj is None:
        return "No file uploaded."
    filepath = file_obj.name                                # Get file path
    if filepath.endswith(".txt"):                           # Handle text file (.txt)
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    elif filepath.endswith(".docx"):                        # Handle Word document (.docx)
        doc = docx.Document(filepath)
        content = "\n".join([para.text for para in doc.paragraphs])
        return content if content.strip() else "(The document is empty.)"
    else:                                                   # Unsupported format
        return "Unsupported file format. Please upload .txt or .docx."

view = gr.Interface(
                    fn=read_file,
                    inputs=gr.File(label="Upload a .txt or .docx file"),
                    outputs=gr.Textbox(label="File Content", lines=15),
                    title="Use of File Upload Component in Gradio",
                    description="Select a file (.txt or .docx) from your computer:",
                    flagging_mode="never"
)
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 6: Q/A Bot using OpenAI's Responses API (Local/Cloud Models via Ollama)</span>

In [28]:
from openai import OpenAI
import gradio as  gr

# The OpenAI client defaults to OpenAI’s servers,so you must specify the base_url to the localhost where ollama server is running
client = OpenAI(base_url="http://localhost:11434/v1", api_key="abc") 

def ask_ai(user_question):
    response = client.responses.create(
                                        model="llama3.2:latest",    # "llama3.2:latest", "llama3.2:1b", "deepseek-r1:1.5b", "mygemma", "deepseek-v3.1:671b-cloud", # "gpt-oss:20b-cloud", "qwen3-coder:480b-cloud"
                                        input=[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": user_question}]
                                    )  
    return response.output_text

view = gr.Interface(
                    fn=ask_ai,                      # function to call
                    inputs=gr.Textbox(label="Ask a question"),  # single text input
                    outputs=[gr.Textbox(label="Model Response:", lines=15)], # outputs=[gr.Markdown(label="Model Response:", lines=15)],
                    title="Q/A Bot using OpenAI's Responses API (Local Models via Ollama)",
                    flagging_mode="never"
)

view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 7: Q/A Bot using OpenAI's Responses API (Groq Hosted Models)</span>

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as  gr

# Load GROQ API key from .env
load_dotenv("../keys/.env", override=True)
groq_api_key = os.getenv("GROQ_API_KEY")


# The OpenAI client defaults to OpenAI’s servers,so you must specify the base_url to Groq’s OpenAI-compatible API endpoint (when using a Groq API key with the OpenAI client).
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) 


def ask_ai(user_question):
    response = client.responses.create(
                                        model="openai/gpt-oss-20b",   # verified live on Groq; the old llama-4-maverick tag now 404s
                                        input=[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": user_question}]
                                    )  
    return response.output_text


# Simple Interface (not ChatInterface)
view = gr.Interface(
                    fn=ask_ai,                      # function to call
                    inputs=gr.Textbox(label="Ask a question"),  # single text input
                    outputs=[gr.Textbox(label="Model Response:", lines=15)], # outputs=[gr.Markdown(label="Model Response:", lines=15)],
                    title="Q/A Bot using OpenAI's Responses API (Groq Hosted Models)",
                    flagging_mode="never"
)

view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 8: A Q/A Bot using Llama3.2 running on Local Box and Claude Haiku 4.5</span>

In [1]:
import gradio as gr
from openai import OpenAI
import anthropic
import os
from dotenv import load_dotenv

# Code to Access Llama3.2 via OpenAI Chat Completion API
client_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def ask_llama(user_prompt):
    response = client_openai.chat.completions.create(
                                                model='llama3.2',
                                                messages=[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': user_prompt}]
                                                )
    return response.choices[0].message.content


# Code to Access Claude-3-Haiku 
load_dotenv('../keys/.env', override=True) 
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
client_claude = anthropic.Anthropic(api_key=anthropic_api_key)
def ask_claude(messages):
    messages = [{"role": "user", "content": messages}]
    response = client_claude.messages.create(
                                    messages=messages,
                                    system="You are a helpful assistant that provides concise answers.",
                                    model="claude-haiku-4-5-20251001",   # claude-3-haiku-20240307 is retired
                                    max_tokens=1024
    )    
    return response.content[0].text



# Function to be called by Gradio
def select_model(prompt, model):
    if model=="Llama":
        result = ask_llama(prompt)
    elif model=="Claude":
        result = ask_claude(prompt)
    return result


view = gr.Interface(
    fn=select_model,
    inputs=[gr.Textbox(label="Your question:"), gr.Dropdown(["Llama", "Claude"], label="Select model", value="Llama")],
    outputs=[gr.Textbox(label="Model Response:", lines=10)],
    title="Q/A Bot using Llama3.2 running on Local Box and Claude Haiku 4.5",
    flagging_mode="never"
)

view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 9: A Q/A Bot that Answers from a File Contents (Local/Cloud Models via Ollama)</span>

In [29]:
from openai import OpenAI
import gradio as gr
import docx  # for reading .docx files

# Initialize OpenAI client (Llama local API)
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# ---------------------------------
# Function to read uploaded file
# ---------------------------------
def read_file(file_obj):
    if file_obj is None:
        return ""
    filepath = file_obj.name
    # Handle text file (.txt)
    if filepath.endswith(".txt"):
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    # Handle Word document (.docx)
    elif filepath.endswith(".docx"):
        doc = docx.Document(filepath)
        content = "\n".join([para.text for para in doc.paragraphs])
        return content if content.strip() else "(The document is empty.)"
    # Unsupported format
    else:
        return "Unsupported file format. Please upload .txt or .docx."

# ------------------------------------------
# Function to ask Llama with file context
# ------------------------------------------
def ask_ollama(user_prompt: str, file_obj=None):
               #developer_prompt: str = "You are a helpful assistant.",
               #model: str = "llama3.2",
               #temperature: float = 0.0):
    
    # Read file content and combine system prompt and file content
    file_content = read_file(file_obj) 
    if file_content:
        system_message = f"You are a helpful assistant, who always answers from the file contents that is given to you.\n\nHere is the context from the uploaded file:\n{file_content}"
    # Make the call to the model
    response = client.responses.create(
                                    model="gpt-oss:20b-cloud",    # "llama3.2:latest", "llama3.2:1b", "deepseek-r1:1.5b", "mygemma", "deepseek-v3.1:671b-cloud", # "gpt-oss:20b-cloud", "qwen3-coder:480b-cloud"
                                    input=[{"role": "system", "content": system_message}, {"role": "user", "content": user_prompt}],
                                    temperature=0.3,                      # Low temperature for more focused/deterministic responses
    )
    return response.output_text

# ------------------
# Gradio interface
# ------------------
view = gr.Interface(
                    fn=ask_ollama,
                    inputs=[
                            gr.Textbox(label="Your question:"),
                            gr.File(label="Upload a .txt or .docx file")
                            ],
                    outputs=[gr.Textbox(label="Model Response:", lines=15)],
                    title="Q/A Bot that Answers from a File Contents (Local/Cloud Models via Ollama)",
                    description="Upload a .txt or .docx file and ask questions based on its content.",
                    flagging_mode="never"
                )

view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 10: A Research Paper Summarizer (Local/Cloud Models via Ollama)</span>

In [1]:
from openai import OpenAI
import gradio as gr
import docx  # (not used yet, but kept for future extension)

# Initialize OpenAI client for Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# ---------------------------------------------------------
# Function to generate summary
# ---------------------------------------------------------
def summarize_paper(paper_input, style_input, length_input):
    prompt = f"Explain the research paper titled {paper_input}. Explanation Style: {style_input} Explanation Length: {length_input}. Structure the response clearly and make it easy to understand."
    response = client.chat.completions.create(
                                            model="llama3.2",   # Ensure this exists via: ollama list
                                            messages=[
                                                        {"role": "system", "content": "You are a helpful AI research assistant."},
                                                        {"role": "user", "content": prompt}
                                                    ]
                                            )
    summary = response.choices[0].message.content
    return prompt.strip(), summary.strip()

# ---------------------------------------------------------
# Build Gradio interface
# ---------------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("<h1 style='text-align: center;'>🧠 Research Paper Summarizer</h1>")
    gr.Markdown("## Select a research paper and customize the style and length of explanation.")

    # Layout Row
    with gr.Row():
        # -----------------------
        # Left Column: Inputs
        # -----------------------
        with gr.Column():

            paper_input = gr.Dropdown(
                choices=[
                    "Attention Is All You Need",
                    "BERT: Pre-training of Deep Bidirectional Transformers",
                    "GPT-3: Language Models are Few-Shot Learners",
                    "Diffusion Models Beat GANs on Image Synthesis"
                ],
                label="Select Research Paper",
                value="Attention Is All You Need"
            )

            style_input = gr.Dropdown(
                choices=[
                    "Beginner-Friendly",
                    "Technical",
                    "Code-Oriented",
                    "Mathematical"
                ],
                label="Explanation Style",
                value="Beginner-Friendly"
            )

            length_input = gr.Dropdown(
                choices=[
                    "Short (1-2 paragraphs)",
                    "Medium (3-5 paragraphs)",
                    "Long (detailed explanation)"
                ],
                label="Explanation Length",
                value="Medium (3-5 paragraphs)"
            )

            summarize_btn = gr.Button("Summarize", variant="primary")
        # -----------------------
        # Right Column: Outputs
        # -----------------------
        with gr.Column():

            prompt_output = gr.Textbox(
                label="Prompt Sent to Model",
                lines=6
            )

            summary_output = gr.Textbox(
                label="Summary",
                lines=15
            )
    # Button Action
    summarize_btn.click(
        fn=summarize_paper,
        inputs=[paper_input, style_input, length_input],
        outputs=[prompt_output, summary_output]
    )

demo.launch(share=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 11: A Q/A Bot that Extracts a youtube video transcript and Answers from the Contents (Local/Cloud Models via Ollama)</span>

## Step 1 (Generating YouTube Video Transcript)
#### https://www.youtube.com/watch?v=ndl79-VDl50
- You may have to install the `youtube-transcript-api` using this command: `uv add youtube-transcript-api`

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi    # YouTube Transcript API is used to fetch video transcripts

# NOTE: only the LAST assignment takes effect - the earlier ones are alternatives
# you can try by moving the one you want to the bottom (or commenting out the rest).
# video_id = "ndl79-VDl50"    # https://www.youtube.com/watch?v=ndl79-VDl50
# video_id = "KMHkbXzHn7s"
# video_id = "IuyVVtr1uhY"
video_id = "skut_4jdvzc"      # <- the video actually used below
api = YouTubeTranscriptApi()                                  # Create an instance of the YouTubeTranscriptApi class
transcript_list = api.list(video_id)                          # Get the list of all available transcripts for the video (different languages, auto-generated, etc.)
transcript = transcript_list.find_transcript(['en'])          # Find and select the English (en) or Urdu (ur) transcript from the available transcripts
transcript_data = transcript.fetch()                          # Fetch the actual transcript data (returns a list of FetchedTranscriptSnippet objects). Each snippet contains: text (the spoken words), start (timestamp), and duration
transcript_text = " ".join([t.text for t in transcript_data]) # Extract 'text' attribute from transcript_data and then join with spaces
print(transcript_text) 

The AI world loves buzzwords. New terms are popping up faster than most people can track, and it's getting hard to tell which ones actually mean something and which are just noise. That confusion is a problem. When everything sounds important, nothing does. But, a few of these terms are worth knowing because they genuinely build on each other in a useful way. Prompt engineering, context engineering, harness engineering, and loop engineering. So, today, instead of adding to the noise, I'm going to walk you through all four with simple examples so that by the end, you'll actually know what people are talking about. Prompt engineering, in simple words, is just how to talk to AI. And, like any conversation, how you frame it matters as much as what you ask. Quick example, I'll open AI chat and turn off code-based mode so the model has zero access to my project Netflix movies analyzer. This is purely for demo reasons. I want to show prompt engineering on its own without any other concept int

>- Can you summarize the video?
>- What are the major topics discussed in this video? Just list them
>- This course is designed for which students category?
>- How binary software packages are installed?
>- Which programming tools are used to understand system calls like fork, wait, and exit?
>- What inter-process communication methods are discussed in the lecture?
>- What are character special files and block special files in Linux?
>- How does the Linux kernel manage information about running processes?
>- What is the capital of Pakistan?
>- What is the name of the instructor?
>- Who developed the Linux kernel?
>- What is the difference between Linux and Windows process management?

## Step 2 (Send the Transcript to AI Assistant and Ask Qs via Gradio Interface)

In [31]:
from openai import OpenAI
import gradio as gr

# Initialize OpenAI client (Llama local API)
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

rules = """You are a helpful assistant that answers questions based ONLY on the provided youtube transcribed content. 
IMPORTANT RULES:
1. You must ONLY use information that is explicitly stated in the provided youtube transcript
2. If the information needed to answer a question is NOT available in the provided content, you must say "I cannot answer this question as the information is not available in the provided in the youtube transcription provided to me."
3. Do not use any external knowledge or make assumptions beyond what is stated in the youtube transcribed content.
4. Be specific and cite relevant parts of the content when answering
5. If you're unsure whether information is in the content, err on the side of caution and say the information is not available
6. Keep your answers concise and directly relevant to the question asked

Remember: Your knowledge is limited to ONLY what appears in the youtube transcribed content provided to you."""



# Function to ask Llama with file context
def ask_ollama(user_prompt: str):
    # Combine system prompt and file content
    system_message = rules
    system_message += f"\n\nHere is the context from the transcribed youtube video:\n{transcript_text}"
    # Make the chat completion call
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ],
        model="llama3.2",
        temperature=0.0
    )
    return response.choices[0].message.content



# Simple Interface (not ChatInterface)
view = gr.Interface(
    fn=ask_ollama,                      # function to call
    inputs=gr.Textbox(label="Ask a question"),  # single text input
    outputs=gr.Textbox(label="AI Response", lines=15),
    title="Q/A Bot with Llama2 from YouTube Video",
    flagging_mode="never"
)
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >Gradio Example 12: A Q/A Bot that Answers from a Webpage (Local/Cloud Models via Ollama)</span>

In [17]:
# Scraping content of a web page
import requests  
from bs4 import BeautifulSoup   
import urllib3  
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) # Disable SSL warnings when verify=False is used (optional but recommended)


url="https://arifbutt.me"
#url="https://quotes.toscrape.com"
# Define headers to mimic a real browser request (helps avoid being blocked)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"}
response = requests.get(url, headers=HEADERS, timeout=10, verify=False) # Send GET request to the URL with headers and 10 second timeout

soup = BeautifulSoup(response.content, 'html.parser') # Parse the HTML content using BeautifulSoup with html.parser

text_content = ""                   # Initialize empty string to store cleaned text content
if soup.body:                       # Check if the body tag exists
    for tag in soup.body(["script", "style", "img", "input"]):     # Remove irrelevant tags (script, style, img, input) that don't contain useful text
        tag.decompose()                                            # Completely remove these tags from the soup object
    text_content = soup.body.get_text(separator="\n", strip=True)  # Extract all text from body, separate with newlines, and strip whitespace
text_content

"Skip to content\nDr. Arif Butt\nHome\nCourses\nPublications\nResearch\nCV\nContact\nHome\nCourses\nPublications\nResearch\nCV\nContact\nDr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.\nWith over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.\nBeyond academia, he is a technology entrepreneur, serving as the Found

In [1]:
import os                          
import openai                      
import requests  
from bs4 import BeautifulSoup     
import urllib3                   

import gradio as gr                
from openai import OpenAI          
from dotenv import load_dotenv     
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) # Disable SSL warnings when verify=False is used (optional but recommended)


###################################################################
# Select the Model to use
###################################################################
MODEL= 'llama3.2' #   # "llama3.2:latest", "llama3.2:1b", "deepseek-r1:1.5b", "mygemma", "deepseek-v3.1:671b-cloud", # "gpt-oss:20b-cloud", "qwen3-coder:480b-cloud"
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


###################################################################
# Function to scrape webpage content
###################################################################
# Define headers to mimic a real browser request (helps avoid being blocked)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"}

def get_webpage_data(url):
    response = requests.get(url, headers=HEADERS, timeout=10, verify=False) # Send GET request to the URL with headers and 10 second timeout
    soup = BeautifulSoup(response.content, 'html.parser') # Parse the HTML content using BeautifulSoup with html.parser
    text_content = ""                   # Initialize empty string to store cleaned text content
    if soup.body:                       # Check if the body tag exists
        for tag in soup.body(["script", "style", "img", "input"]):     # Remove irrelevant tags (script, style, img, input) that don't contain useful text
            tag.decompose()                                            # Completely remove these tags from the soup object
        text_content = soup.body.get_text(separator="\n", strip=True)  # Extract all text from body, separate with newlines, and strip whitespace
    return text_content


###################################################
# Function to ask questions about the webpage content
###################################################
def ask_question_about_webpage(website_content, question):
    # Define system prompt that instructs the model to only use provided content
    system_prompt = """You are a helpful assistant that answers questions based ONLY on the provided website content. 
IMPORTANT RULES:
1. You must ONLY use information that is explicitly stated in the provided website content
2. If the information needed to answer a question is NOT available in the website content, you must say "I cannot answer this question as the information is not available in the provided website content."
3. Do not use any external knowledge or make assumptions beyond what is stated in the website
4. Be specific and cite relevant parts of the content when answering
5. If you're unsure whether information is in the content, err on the side of caution and say the information is not available
6. Keep your answers concise and directly relevant to the question asked

Remember: Your knowledge is limited to ONLY what appears in the website content provided to you."""

    user_prompt = f"""               # Create user prompt with website content and the question
        Website Content: {website_content}
        Question: {question}
        Please answer based ONLY on the website content above. If the information is not available, state that clearly.
        """
    response = client.responses.create(
                                    model=MODEL,  
                                    input=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                                    temperature=0.3,                      # Low temperature for more focused/deterministic responses
    )
    return response.output_text

###################################################
# Main function that combines scraping and Q&A
###################################################
def scrape_and_ask(url, question):
    if not url:
        return "Please enter a valid URL."
    webpage_data = get_webpage_data(url)                          # Scrape the webpage content
    answer = ask_question_about_webpage(webpage_data, question)   # Ask the question using the scraped content
    return answer


###################################################
# Create and launch Gradio web interface
###################################################
interface = gr.Interface(
                        fn=scrape_and_ask,                       # Function to call when user submits
                        inputs=[                                 # Define input components
                                gr.Textbox(label="Enter a URL:"),                    # Textbox for URL input
                                gr.Textbox(label="Ask a question about the webpage:") # Textbox for question input
                                ],
                        outputs=[gr.Textbox(label="Answer:", lines=15)],
                        title="Webpage Q/A Bot",                 # Interface title
                        description="Enter a webpage URL and ask questions based solely on the webpage content.",  # Description
                        flagging_mode="never"                    # Disable flagging feature
                        )
interface.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# TO DO:
### [Watch my basic scrapping videos using BS and Selenium](https://www.youtube.com/watch?v=T2FCN1gPcyE&list=PL7B2bn3G_wfCODLEzjCrpq-GsstBFcygc&index=1)
### Common Web Scraping Challenges:
* Bot detection systems like **[User-Agent checks](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/User-Agent)** and request rate monitoring.
* **[CAPTCHAs](https://en.wikipedia.org/wiki/CAPTCHA)** designed to block automated scripts.
* Legal and ethical concerns, such as:
  * Violating **[Terms of Service](https://en.wikipedia.org/wiki/Terms_of_service)**.
  * Copyright and data ownership issues.
  * Overloading websites with too many requests.
* Ethical scraping requires:
  * Respecting **[robots.txt](https://developers.google.com/search/docs/crawling-indexing/robots/intro)** rules.
  * Using polite request rates.
  * Avoiding unauthorized access.

### Why Traditional Scraping Fails
* Many modern websites block bots using:
  * **[Browser fingerprinting](https://en.wikipedia.org/wiki/Device_fingerprint)**
  * **[IP blacklisting](https://www.geeksforgeeks.org/computer-networks/what-is-ip-blocklisting/?utm_source=chatgpt.com)**
  * **[JavaScript rendering](https://developer.mozilla.org/en-US/docs/Web/JavaScript)** (content loads dynamically)
  * Login / session walls
* Manually bypassing CAPTCHAs using tools like **[Selenium](https://www.selenium.dev/)** or **[Playwright](https://playwright.dev/)** adds:
  * Complexity
  * Maintenance overhead
  * Slow performance

### Use [Bright Data Web Unlocker](https://brightdata.com/products/web-unlocker)
- Bright Data Web Unlocker helps you scrape websites that normally block bots by providing:
    - Automatic CAPTCHA solving
    - Smart IP rotation
    - Realistic browser fingerprinting
    - Built-in JavaScript rendering
- It also supports:
    - Geo-targeting (access websites from different countries)
    - Custom headers
    - Cookies
    - Access to premium / protected domains
- This makes it useful for scraping difficult websites that use CAPTCHA, login walls, and bot detection like Goodreads, Walmart, Instagram, and LinkedIn


# Programming Assignment 01:

<h2 align="center"><div class="alert alert-success" style="margin: 20px">You will build one application in six increasing versions. Each version is developed <b>locally</b>, pushed to <b>GitHub</b>, and deployed live on <b>Render</b>.</div></h2>

## The Deployment Workflow: Local → GitHub → Render

Every version follows the same three stages. Learn this once and it applies to all six.

### Stage 1 — Build and run it locally
Your project folder must look like this:

```
MSDSF25M001-ver1/
├── app.py              # your Gradio application
├── requirements.txt    # pinned dependencies
├── .env                # your API keys - NEVER committed
├── .gitignore          # must list .env
└── README.md           # description + live URL + screenshot
```

```bash
uv add gradio openai python-dotenv     # or: pip install ...
python app.py                          # open http://localhost:7860 and test it
```

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Rule: the app must work locally <b>before</b> you push it anywhere. Most "deployment failures" are ordinary bugs that were already present.</div></h3>

### Stage 2 — Push to GitHub
```bash
git init
git add .                              # .gitignore keeps .env out of the commit
git commit -m "Version 1: scraper + Q&A bot"
git branch -M main
git remote add origin https://github.com/<your-username>/MSDSF25M001-ver1.git
git push -u origin main
```

<h3 align="center"><div class="alert alert-danger" style="margin: 20px">⚠️ After pushing, <b>open your repository on GitHub and confirm <code>.env</code> is not there.</b> A public repo containing an API key is scraped by bots within minutes and the key will be abused at your expense.</div></h3>

Your `.gitignore` must contain at least:
```
.env
__pycache__/
*.pyc
.venv/
```

### Stage 3 — Deploy on Render
1. Sign up at [render.com](https://render.com) (free, GitHub login works).
2. **New → Web Service** → connect your GitHub repository.
3. Configure:

| Setting | Value |
|---|---|
| **Language / Runtime** | `Python 3` |
| **Build Command** | `pip install -r requirements.txt` |
| **Start Command** | `python app.py` |
| **Instance Type** | `Free` |
| **Environment Variables** | Add `GROQ_API_KEY` (and any others) here — **not** in your repo |

4. Click **Deploy**. Watch the logs; when it says *"Your service is live"*, open the URL.
5. Every later `git push` redeploys automatically.

### Three code changes Render requires

**1. Bind to Render's port — this is the #1 cause of failed deployments.**
```python
import os

demo.launch(
    server_name="0.0.0.0",                              # not 127.0.0.1
    server_port=int(os.environ.get("PORT", 7860)),      # Render injects PORT
)
```
If you hard-code `7860` or omit `server_name`, Render reports *"no open ports detected"* and the deploy fails.

**2. Read secrets from the environment** — the same code works locally (from `.env`) and on Render (from the dashboard):
```python
import os
from dotenv import load_dotenv

load_dotenv()                       # no-op on Render, loads .env locally
api_key = os.getenv("GROQ_API_KEY")
```

**3. Pin your dependencies** so a future library release cannot break a working deployment:
```
gradio==5.49.1
openai==2.14.0
python-dotenv==1.0.1
```

### Know the free tier's limits (these are not bugs)

| Limit | What it means for you |
|---|---|
| **Sleeps after 15 min idle** | The first request after a pause takes ~1 minute to wake. Your app is not broken — mention this in your README. |
| **Ephemeral filesystem** | Anything written to disk is **lost** on redeploy or spin-down. This directly affects Version 3 — see the note there. |
| **750 instance-hours / month** | Plenty for six lightly-used student projects, because idle services sleep. Do not keep them artificially awake. |
| **512 MB RAM** | Enough for a Gradio app that *calls* an API. **Never load a model into the web service** — always use a hosted API (Groq, OpenAI, HF Inference). |

---

## Deliverables for every version

For each version you must submit **two links**:

1. A **public GitHub repository** named `MSDSF25M001-ver1` (…`ver2`, …`ver6`) containing `app.py`, `requirements.txt`, `README.md`, and `.gitignore`.
2. The **live Render URL** of the deployed app.

Your `README.md` must contain: a one-paragraph description, the live Render URL, a screenshot of the running app, and local setup instructions.

<h3 align="center"><div class="alert alert-danger" style="margin: 20px">Any repository containing a committed API key scores <b>zero</b> for that version. Rotate the key immediately if you leak one.</div></h3>

---

## Version 1 — Scrape a Bot-Protected Website and Perform Q&A using an AI Assistant

- Extend the given BeautifulSoup + Requests starter project to scrape a bot-protected website like [https://www.goodreads.com](https://www.goodreads.com), which employs CAPTCHA and bot-detection mechanisms that block standard HTTP requests.
- Replace the normal `requests.get()` call with the [Bright Data Web Unlocker API](https://brightdata.com/products/web-unlocker), which handles bot bypass on your behalf. You must sign up for a Bright Data account, obtain your API credentials, and pass them correctly in your request headers.
- Fetch the full HTML content of a protected page (e.g., the Goodreads Best Books list).
- Parse the returned HTML using **BeautifulSoup** to extract structured data fields: **Book Title**, **Author Name**, and **Star Rating** for each listed book.
- Store the extracted data as a list of dictionaries (or a Pandas DataFrame) and inject it as context into your Q&A bot's system prompt, so the model can answer questions such as:
  - *"What is the rating of the top-ranked book on this page?"*
  - *"Who wrote the second book in the list?"*
- The Q&A bot must accept free-text user queries via a **Gradio** `ChatInterface` and respond using an LLM hosted on **Groq Cloud**.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver1</b> + its live Render URL.
</h4>

---

## Version 2 — Generate a YouTube Video Transcript and Perform Q&A using an AI Assistant
- This version extends the Version 1 app so that it supports **two independent input modes**, selectable by the user via a Gradio Tab interface:
- **Tab 1:** Bot-Protected Website Scraper (carry forward from Version 1):
  - User enters any URL in a text box.
  - The app fetches and parses the page using Bright Data Web Unlocker API + BeautifulSoup, exactly as in Version 1.
  - Extracted content is used as context for Q&A.
- **Tab 2:** YouTube Transcript Q&A:
  - User enters a valid **YouTube Video ID** (e.g., `dQw4w9WgXcQ`) in a text box — not the full URL.
  - The app uses the [`youtube-transcript-api`](https://pypi.org/project/youtube-transcript-api/) Python library to fetch the auto-generated or manual transcript for that video.
  - The full transcript text is passed as context to a Groq-hosted LLM, which then answers user questions about the video content, such as:
    - *"What is the main topic of this video?"*
    - *"Summarize the video in 3 bullet points."*
  - If no transcript is available for the given video ID, the app must display a clear error message to the user.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver2</b> + its live Render URL. All features from Version 1 must be carried forward and must remain fully functional.
</h4>

---

## Version 3 — Multi-Turn AI Chatbot with Persistent Storage
- Modern AI assistants now include memory features that persist across conversations. So your AI assistant can:
    - Remember personal preferences: Your communication style, preferred formats, frequently referenced topics.
    - Retain context: Details from past projects, key contacts, recurring tasks .
    - Store explicit instructions: Custom rules you’ve given the AI, like “always respond formally” or “cite sources when summarizing research.”
- This version adds three major enhancements on top of Version 2:
- **Multi-Turn Conversation (Short-term/Session Memory):**
    - Maintain a `conversation_history` list (in the format `[{"role": "user"/"assistant", "content": "..."}]`) for the duration of a single user session.
    - Every new user message must append the full history to the API call so the model has context of previous turns. The bot should be able to answer follow-up questions like *"Tell me more about the second book"* without the user repeating context.
- **Persistent Storage (Cross-Session Memory):**
    - Use a local **JSON file** (e.g., `chat_history.json`) to persist conversation history across app restarts.
    - On startup, load existing history from storage and display it in the chat interface.
    - After each turn, immediately write the updated history to storage.

> ⚠️ **Render's filesystem is ephemeral.** Files written by your app are erased whenever the service redeploys or spins down after 15 minutes of inactivity. So:
> - **Demonstrate true cross-session persistence locally** (stop the app, restart it, show the history survived) — record this in your README with a screenshot.
> - On Render, the same code will persist history only for as long as the instance stays awake. State this limitation in your README.
> - **Optional (for extra credit):** replace the JSON file with a free hosted database such as [Supabase](https://supabase.com) or [Neon](https://neon.tech) Postgres, which survives redeploys.
- **Editable User Preferences:**
    - It also make the storage editable by adding user preferences at run-time by the user.
    - These preferences must be injected into the system prompt dynamically on every API call.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver3</b> + its live Render URL. All features from Version 2 must be carried forward and must remain fully functional.
</h4>

---

## Version 4 — Multimodal AI Assistant (Voice Input & Audio Output)
- This version extends Version 3 into a **fully voice-enabled AI assistant** with the following capabilities:
    - **Voice Input (Speech-to-Text):**
        - Add a Gradio `Audio` component (with `source="microphone"`) that allows the user to record their query by speaking.
        - Transcribe the recorded audio using a **speech-to-text model**. Use one of the following:
        - **Closed-source (via Groq Cloud):** `whisper-large-v3` or `whisper-large-v3-turbo`
        - **Open-source (run locally or via Hugging Face Inference API):** [`openai/whisper-base`](https://huggingface.co/openai/whisper-base) or [`openai/whisper-medium`](https://huggingface.co/openai/whisper-medium)
        - The transcribed text is then passed to the LLM exactly as a typed message would be.
    - **Voice Output (Text-to-Speech):**
        - Convert the LLM's text response to audio using a **text-to-speech model**. Use one of the following:
        - **Closed-source:** OpenAI TTS API (`tts-1` or `tts-1-hd`) with voices like `alloy` or `nova`
        - **Open-source (via Hugging Face):** [`microsoft/speecht5_tts`](https://huggingface.co/microsoft/speecht5_tts) or [`suno/bark`](https://huggingface.co/suno/bark)
        - Play back the generated audio response automatically in the Gradio interface using a Gradio `Audio` output component.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver4</b> + its live Render URL. All features from Version 3 must be carried forward and must remain fully functional.
</h4>

---

## Version 5 — Multimodal AI Assistant (Image Generation)
- This version extends Version 4 by adding **on-demand image generation** as an additional output modality.
- **Image Generation Logic:**
    - After the LLM generates a text response, use a secondary LLM call (or a simple keyword/intent check) to decide whether the response would benefit from a visual illustration (e.g., if the user asked *"Explain the solar system visually"* or *"Show me what a transformer architecture looks like"*).
    - If an image is warranted, automatically generate one using a **text-to-image model** and display it alongside the text response in the Gradio interface.
    - Use one of the following models:
        - **Closed-source:** [`gpt-image-1`](https://platform.openai.com/docs/guides/images) via the OpenAI Images API (`dall-e-3` has been retired)
        - **Open-source (via Hugging Face Inference API):** [`stabilityai/stable-diffusion-xl-base-1.0`](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) or [`black-forest-labs/FLUX.1-schnell`](https://huggingface.co/black-forest-labs/FLUX.1-schnell) — both are free and state-of-the-art open-source options
    - The image prompt sent to the image model should be a **condensed, descriptive version** of the user's query (not the full LLM response). You may use a short LLM call to generate this image prompt automatically.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver5</b> + its live Render URL. All features from Version 4 must be carried forward and must remain fully functional.
</h4>

---
## Version 6 — Multi-Speaker Meeting Capture & Automated Minutes Agent
- This version extends Version 5 by enabling the assistant to capture and transcribe multi-speaker meeting audio involving structured roles such as President and Secretary using speaker diarization.
- Once the meeting ends, the agent must automatically generate formal meeting minutes in a predefined organizational format including agenda items, decisions taken, action items, and assigned responsibilities.
- The generated minutes are first emailed to the Secretary for review and verification via an automated email workflow integrated into the app.
- After approval by the Secretary, the agent must automatically distribute the finalized minutes via email to all meeting members for reading prior to official signing.
- This version introduces an end-to-end Agentic workflow combining speech processing, role-aware summarization, document generation, and human-in-the-loop approval.

<h4 align="left" style="color: purple;">
- Deliverable: GitHub repo <b>MSDSF25M001-ver6</b> + its live Render URL. All features from Version 5 must be carried forward and must remain fully functional.
</h4>

---
## Suggested Models
| Task                                                                | Closed-Source Models (Vendor API)                                                                                                                                                                                                                                                                                                       | Open-Source / Hosted Models                                                                                                                                                                                                                                                                                                                 |
| ------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Ver 1 & 2 — Text Q&A**                                            | [`gpt-4o-mini`](https://platform.openai.com/docs/models/gpt-4o-mini) (OpenAI API — cheapest GPT option 💰), [`claude-haiku-4-5`](https://docs.anthropic.com/en/docs/about-claude/models) (Anthropic API — fast & affordable), [`gemini-flash-latest`](https://ai.google.dev/gemini-api/docs/models) (Google AI API — generous free tier ✅) | [`meta-llama/Llama-3.1-8B-Instruct`](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) (Hugging Face), `openai/gpt-oss-20b` ([Groq](https://console.groq.com) — very fast, free tier), locally via [Ollama](https://ollama.com)                                                                                                    |
| **Ver 3 — Multi-Turn Chat**                                         | [`gpt-4o`](https://platform.openai.com/docs/models/gpt-4o) (OpenAI API), [`claude-sonnet-4-6`](https://docs.anthropic.com/en/docs/about-claude/models) (Anthropic API — current & highly capable), [`gemini-pro-latest`](https://ai.google.dev/gemini-api/docs/models) (Google AI API — long context window 🔑)                            | [`meta-llama/Llama-3.3-70B-Instruct`](https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct) (Hugging Face), `openai/gpt-oss-120b` ([Groq](https://console.groq.com) — free hosted), locally via [Ollama](https://ollama.com)                                                                                                        |
| **Ver 4 — Speech-to-Text**                                          | [`whisper-1`](https://platform.openai.com/docs/guides/speech-to-text) (OpenAI API), [`nova-3`](https://developers.deepgram.com/docs/models-overview) (Deepgram API — free $200 credit for new users ✅)                                                                                                                                  | [`openai/whisper-large-v3`](https://huggingface.co/openai/whisper-large-v3) (Hugging Face), [`openai/whisper-large-v3-turbo`](https://huggingface.co/openai/whisper-large-v3-turbo) (Hugging Face — faster & lighter), `whisper-large-v3-turbo` ([Groq](https://console.groq.com) — free hosted), locally via [Ollama](https://ollama.com)  |
| **Ver 4 — Text-to-Speech**                                          | [`tts-1`](https://platform.openai.com/docs/guides/text-to-speech) / [`tts-1-hd`](https://platform.openai.com/docs/guides/text-to-speech) (OpenAI API), [`aura-asteria-en`](https://developers.deepgram.com/docs/tts-models) (Deepgram API — free tier available ✅)                                                                      | [`microsoft/speecht5_tts`](https://huggingface.co/microsoft/speecht5_tts) (Hugging Face), [`suno/bark`](https://huggingface.co/suno/bark) (Hugging Face — expressive, supports emotion), locally via [Ollama](https://ollama.com)                                                                                                           |
| **Ver 5 — Image Generation**                                        | [`gpt-image-1`](https://platform.openai.com/docs/guides/images) (OpenAI API — `dall-e-3` has been retired), [`imagen-3.0-generate-002`](https://ai.google.dev/gemini-api/docs/imagen) (Google AI API — free tier ✅)                                                                                                                                                      | [`stabilityai/stable-diffusion-xl-base-1.0`](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) (Hugging Face), [`black-forest-labs/FLUX.1-schnell`](https://huggingface.co/black-forest-labs/FLUX.1-schnell) (Hugging Face — fastest FLUX variant), locally via [Ollama](https://ollama.com)                                 |
| **Ver 6 — Multi-Speaker Meeting Capture & Automated Minutes Agent** | [`gpt-4o`](https://platform.openai.com/docs/models/gpt-4o) (structured minutes generation), [`whisper-1`](https://platform.openai.com/docs/guides/speech-to-text) (speaker-aware transcription), [`SendGrid`](https://docs.sendgrid.com/) or [`Resend`](https://resend.com/docs) (email automation APIs) | [`pyannote.audio`](https://huggingface.co/pyannote/speaker-diarization) (speaker diarization), [`openai/whisper-large-v3`](https://huggingface.co/openai/whisper-large-v3) (multi-speaker transcription), [`LangGraph`](https://python.langchain.com/docs/langgraph), [`CrewAI`](https://docs.crewai.com/), [`OpenAI Agents SDK`](https://openai.github.io/openai-agents-python/) (agent workflow orchestration) |

---

# [Four Hundred Meters on Mars](https://www.anthropic.com/features/claude-on-mars)

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Autonomous AI agents are no longer confined to Earth — they’re beginning to explore the farthest frontiers of the solar system. Are you ready to build the ones that lead the way?</div></h3>

<h4 align="left" style="color: purple;">
    
>- Claude on Mars tells the inspiring story of how an AI system helped NASA’s Perseverance rover successfully navigate about 400 meters across the surface of Mars, marking one of the first real examples of AI-assisted decision-making on another planet.
>- Instead of directly controlling the rover, Claude supported scientists by analyzing terrain, identifying safe paths, and helping plan movement more efficiently in a harsh and uncertain environment where communication delays with Earth make real-time control impossible.
>- This collaboration shows how advanced AI systems can act as intelligent partners to humans in exploring the space, conduct science, and solve problems far beyond Earth.

</h4>